In [4]:
ls /Volumes/ExtremeSSD/

SyntaxError: invalid syntax (3499605120.py, line 3)

In [5]:
import scanpy as sc
import pandas as pd
from pathlib import Path
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache



ModuleNotFoundError: No module named 'abc_atlas_access'

In [ ]:
import numpy as np

In [ ]:
download_base = Path('/Volumes/ExtremeSSD/data/abc_atlas')

In [ ]:
abc_cache = AbcProjectCache.from_cache_dir(download_base)

In [ ]:
files = abc_cache.list_expression_matrix_files("WMB-10Xv3")

In [ ]:
files

In [ ]:
isocortex_path = abc_cache.get_file_path("WMB-10Xv3", "WMB-10Xv3-Isocortex-1/raw")

In [ ]:
isocortex_2_path = abc_cache.get_file_path("WMB-10Xv3", "WMB-10Xv3-Isocortex-2/raw")

In [ ]:
iso1 = sc.read_h5ad(isocortex_path, backed="r")

In [ ]:
iso2 = sc.read_h5ad(isocortex_2_path, backed="r")

In [ ]:
genes = iso1.var["gene_symbol"].tolist()

In [ ]:
len(genes)

In [ ]:
genes = pd.Series(genes)

In [ ]:
hits = genes[genes.str.contains("htr", case=False, na=False)].unique()

In [ ]:
hits

In [ ]:
final_v3_meta = pd.read_csv("/Volumes/ExtremeSSD/views/final_v3_meta_dump.csv")

In [ ]:
final_v3_meta

In [ ]:
final_v3_meta_idx = final_v3_meta.set_index("cell_label")

In [ ]:
iso1.obs = iso1.obs.join(
    final_v3_meta_idx,
    on="cell_label"
)

In [ ]:
iso1

In [ ]:
iso1.obs[["class", "subclass", "neurotransmitter"]].head()

In [ ]:
neurons = ~iso1.obs["class"].isin([
    "30 Astro-Epen",
    "31 OPC-Oligo",
    "33 Vascular",
    "34 Immune",
    "32 OEC"
])

In [ ]:
iso1[neurons].obs[["class"]].value_counts()

In [ ]:
iso1_neurons = iso1[neurons].to_memory()

In [ ]:
iso1_neurons

In [ ]:
#Focus on 1, 2, 6, and 7
iso1_neurons.obs["class"].value_counts()

In [ ]:
serotonin_gpcr = [
    "Htr1a","Htr1b","Htr1d","Htr1f",
    "Htr2a","Htr2b","Htr2c",
    "Htr4","Htr5a","Htr5b",
    "Htr6","Htr7"
]

serotonin_ionotropic = ["Htr3a", "Htr3b"]

all_serotonin = serotonin_gpcr + serotonin_ionotropic

In [ ]:
all_serotonin 

In [ ]:
sc.pp.normalize_total(iso1_neurons, target_sum=10000)

In [ ]:
sc.pp.log1p(iso1_neurons)

In [ ]:
iso1_neurons.X.min()

In [ ]:
iso1_neurons.X.max()

In [ ]:
sc.pp.highly_variable_genes(
    iso1_neurons,
    n_top_genes=3000,
    flavor="cell_ranger"
)

In [ ]:
iso1_neurons

In [ ]:
iso1_neurons.var.highly_variable.value_counts()

In [ ]:
iso1_hvg = iso1_neurons[:, iso1_neurons.var.highly_variable].copy()

In [ ]:
iso1_hvg

In [ ]:
iso1_hvg.obs.library_label.value_counts()

In [ ]:
iso1_hvg.obs.anatomical_division_label.value_counts()

In [ ]:
iso1_hvg.obs.cluster.value_counts()

In [ ]:
iso1_hvg.obs.supertype.value_counts()

In [ ]:
iso1_hvg.obs.subclass.value_counts()

In [ ]:
iso1_hvg.obs["class"].value_counts()

In [ ]:
iso1_hvg.obs["neurotransmitter"].value_counts()

In [ ]:
sc.tl.pca(iso1_hvg)

In [ ]:
sc.pp.neighbors(iso1_hvg)

In [ ]:
sc.tl.umap(iso1_hvg)

In [ ]:
iso1_hvg

In [ ]:
iso1_neurons.obsm["X_umap"] = iso1_hvg.obsm["X_umap"]

In [ ]:
sc.pl.umap(iso1_neurons, color="class", save="_iso1_class_umap.pdf")

In [ ]:
major_class = iso1_neurons.obs["class"].astype(str).copy()

keep = [
    "01 IT-ET Glut",
    "02 NP-CT-L6b Glut",
    "06 CTX-CGE GABA",
    "07 CTX-MGE GABA",
]

major_class[~major_class.isin(keep)] = "Other"
iso1_neurons.obs["class_major"] = major_class

In [ ]:
iso1_neurons.obs["class_major"].value_counts()

In [ ]:
sc.pl.umap(iso1_neurons, color="class_major")

In [ ]:
all_serotonin

In [ ]:
sc.pl.umap(
    iso1_neurons,
    color=["Htr2a", "Htr3a", "Htr6", "Htr1a"],
    gene_symbols="gene_symbol",
    vmax="p99",
    cmap="viridis"
)

In [ ]:
sc.pl.umap(
    iso1_neurons,
    color=['Htr1a','Htr1b','Htr1d','Htr1f'],
    gene_symbols="gene_symbol",
    vmax="p99",
    cmap="viridis"
)

In [ ]:
sc.pl.umap(
    iso1_neurons,
    color=['Htr2a','Htr2b','Htr2c'],
    gene_symbols="gene_symbol",
    vmax="p99",
    cmap="viridis"
)

In [ ]:
sc.pl.umap(
    iso1_neurons,
    color=serotonin_ionotropic,
    gene_symbols="gene_symbol",
    vmax="p99",
    cmap="viridis"
)

In [ ]:
sc.pl.umap(
    iso1_neurons,
    color=['Htr4','Htr5a','Htr5b','Htr6','Htr7'],
    gene_symbols="gene_symbol",
    vmax="p99",
    cmap="viridis"
)

In [ ]:
sc.pl.umap(
    iso1_neurons,
    color=["Htr2c", "Htr4"],
    gene_symbols="gene_symbol",
    vmax="p99",
    cmap="viridis"
)

In [ ]:
sc.pl.matrixplot(
    iso1_neurons,
    var_names=["Htr2a", "Htr2c", "Htr4"],
    groupby="cluster",
    gene_symbols="gene_symbol",
    standard_scale="var"
)

In [ ]:
all_serotonin

In [ ]:
sc.pl.matrixplot(
    iso1_neurons,
    var_names=all_serotonin,
    groupby="class_major" if "class_major" in iso1_neurons.obs.columns else "class",
    gene_symbols="gene_symbol",
    standard_scale="var",   # makes each gene comparable across classes (z-score per gene)
    cmap="RdBu_r",
)

In [ ]:
sc.pl.dotplot(
    iso1_neurons,
    var_names=all_serotonin,
    groupby="class_major" if "class_major" in iso1_neurons.obs.columns else "class",
    gene_symbols="gene_symbol",
    standard_scale="var",
    dot_max=0.6, 
    save="_iso1_serotonin_receptor_expression.pdf"# keeps huge dots from dominating
)

In [ ]:
iso1_neurons

In [ ]:
htr2a_mask = iso1_neurons.var["gene_symbol"] == "Htr2a"

In [ ]:
iso1_neurons

In [ ]:
htr2a_mask

In [ ]:
from scipy import sparse

def get_gene_expression(adata, gene, gene_col="gene_symbol"):
    idx = np.where(adata.var[gene_col].values == gene)[0]
    if len(idx) != 1:
        raise ValueError(f"{gene} found {len(idx)} times in var['{gene_col}']")
    
    x = adata.X[:, idx[0]]  # (n_cells, 1) slice
    if sparse.issparse(x):
        return x.toarray().ravel()
    else:
        return np.asarray(x).ravel()


In [ ]:
htr2a_expr = get_gene_expression(iso1_neurons, "Htr2a")

In [ ]:
len(htr2a_expr)

In [ ]:
iso1_neurons.obs["Htr2a_pos"] = htr2a_expr > 0

In [ ]:
iso1_neurons.obs["Htr2a_pos"].value_counts()

In [ ]:
iso1_neurons

In [ ]:
df = iso1_neurons.obs[["class", "subclass", "supertype", "cluster"]].copy()
df["Htr2a_pos"] = htr2a_expr > 0

fraction_htr2a = (
    df.groupby("class")["Htr2a_pos"]
    .mean()
    .sort_values(ascending=False)
)

In [ ]:
fraction_htr2a

In [ ]:
df

In [ ]:
df["class"].value_counts()

In [ ]:
df.groupby("class")["Htr2a_pos"].agg(["mean", "count"]).sort_values(by="mean", ascending=False)

In [ ]:
df.groupby("subclass")["Htr2a_pos"].agg(["mean", "count"]).sort_values(by="mean", ascending=False)[:30]

In [ ]:
df_filtered = (
    df.groupby("subclass")["Htr2a_pos"]
      .agg(["mean", "count"])
      .query("count >= 100")
      .sort_values("mean", ascending=False)
)

In [ ]:
df_filtered

In [ ]:
df_filtered.index.tolist()

In [ ]:
sc.pl.dotplot(
    iso1_neurons,
    var_names=["Htr2a"],
    groupby="subclass",
    gene_symbols="gene_symbol",
    dot_min=0.01,
)

In [ ]:
subclass_counts = iso1_neurons.obs["subclass"].value_counts()

valid_subclasses = subclass_counts[subclass_counts >= 100].index
len(valid_subclasses)

In [ ]:
iso1_neurons_sub = iso1_neurons[
    iso1_neurons.obs["subclass"].isin(valid_subclasses)
].copy()

In [ ]:
sc.pl.dotplot(
    iso1_neurons_sub,
    var_names=["Htr2a"],
    groupby="subclass",
    gene_symbols="gene_symbol",
    dot_min=0.01,
)

In [ ]:
order

In [ ]:
# reuse your existing Htr2a_pos column
order = (
    iso1_neurons_sub.obs
    .groupby("subclass")["Htr2a_pos"]
    .mean()
    .sort_values(ascending=False)
    .index
)

iso1_neurons_sub.obs["subclass"] = pd.Categorical(
    iso1_neurons_sub.obs["subclass"],
    categories=order,
    ordered=True,
)

In [ ]:
sc.pl.dotplot(
    iso1_neurons_sub,
    var_names=["Htr2a"],
    groupby="subclass",
    gene_symbols="gene_symbol",
    dot_min=0.01,
)

In [ ]:
targets = [
    "001 CLA-EPd-CTX Car3 Glut",
    "002 IT EP-CLA Glut",
    "052 Pvalb Gaba",
]

adata_targets = iso1_neurons[
    iso1_neurons.obs["subclass"].isin(targets)
].copy()

adata_targets.obs["subclass"].value_counts()

In [ ]:
adata_targets

In [ ]:
sc.pp.highly_variable_genes(adata_targets, n_top_genes=2000)
sc.tl.pca(adata_targets)
sc.pp.neighbors(adata_targets)
sc.tl.umap(adata_targets)

In [ ]:
sc.pl.umap(adata_targets, color="subclass")

In [ ]:
adata_targets

In [ ]:
sc.pl.violin(
    adata_targets,
    keys="Htr2a",
    groupby="subclass",
    gene_symbols="gene_symbol",
    stripplot=False,
)

In [ ]:
sc.pl.dotplot(
    adata_targets,
    var_names=["Htr2a"],
    groupby="cluster",
    gene_symbols="gene_symbol",
    dot_min=0.01,
)

In [ ]:
sc.pl.dotplot(
    adata_targets,
    var_names=["Htr2a"],
    groupby="supertype",
    gene_symbols="gene_symbol",
    dot_min=0.01,
    save="_claustrom_related_and_pvalb_neurons.pdf"
)

In [ ]:
adata_targets.obs["supertype"].value_counts()

In [ ]:
adata_targets.obs["cluster"].value_counts()

In [ ]:
sc.pl.dotplot(
    adata_targets,
    var_names=["Htr2a"],
    groupby="cluster",
    gene_symbols="gene_symbol",
    dot_min=0.01,
)

In [ ]:
sc.pl.dotplot??

In [ ]:
counts = iso1_neurons.obs["supertype"].value_counts()
valid_supertypes = counts[counts >= 100].index

adata_st = iso1_neurons[
    iso1_neurons.obs["supertype"].isin(valid_supertypes)
].copy()

sc.pl.dotplot(
    adata_st,
    var_names=["Htr2a"],
    groupby="supertype",
    gene_symbols="gene_symbol",
    save="_supertype_dotplot_5htr2a.pdf"
)

In [ ]:
htr2a_expr = get_gene_expression(iso1_neurons, "Htr2a")

In [ ]:
tmp = iso1_neurons.obs[["supertype"]].copy()
tmp["expr"] = htr2a_expr
tmp["pos"] = tmp["expr"] > 0

summary = tmp.groupby("supertype").agg(
    n=("expr", "size"),
    frac_pos=("pos", "mean"),
    mean_pos=("expr", lambda x: x[x > 0].mean() if (x > 0).any() else 0.0),
    mean_all=("expr", "mean"),
).sort_values("frac_pos", ascending=False)



summary[summary["n"] > 100].head(20)

In [ ]:
summary[summary["n"] > 100].head(40)

In [ ]:
def group_layers_from_supertype(st):
    st = str(st)

    if "CLA-EPd-CTX" in st or "EP-CLA" in st:
        return "Claustrum-related IT"

    if " IT " in st and "L2/3" in st:
        return "L2/3 IT"

    if " IT " in st and "L4/5" in st:
        return "L4/5 IT"

    if " IT " in st and "L4" in st:
        return "L4 IT"

    if " IT " in st and "L5" in st:
        return "L5 IT"

    if " IT " in st and "L6" in st:
        return "L6 IT"

    if "Pvalb" in st:
        return "Pvalb"

    if "Sst" in st:
        return "Sst"

    if any(x in st for x in ["Vip", "Lamp5", "Sncg", "CGE"]):
        return "Vip/Lamp5"

    return None

In [ ]:
iso1_neurons.obs["group_6"] = iso1_neurons.obs["supertype"].map(group_layers_from_supertype)

iso1_g6 = iso1_neurons[iso1_neurons.obs["group_6"].notna()].copy()

#iso1_g6.obs[["class", "group_6"]].value_counts()

In [ ]:
iso1_g6

In [ ]:
iso1_g6.obs[["supertype", "group_6"]].value_counts()[:51:]

In [ ]:
iso1_g6.obs["group_6"].value_counts()

In [ ]:
iso1_g6

In [ ]:
htr2a_expr = get_gene_expression(iso1_g6, "Htr2a")

In [ ]:
len(htr2a_expr)

In [ ]:
tmp = iso1_g6.obs[["group_6"]].copy()
tmp["expr"] = htr2a_expr
tmp["pos"] = tmp["expr"] > 0

summary = tmp.groupby("group_6").agg(
    n=("expr", "size"),
    frac_pos=("pos", "mean"),
    mean_pos=("expr", lambda x: x[x > 0].mean() if (x > 0).any() else 0.0),
    mean_all=("expr", "mean"),
).sort_values("frac_pos", ascending=False)



summary

In [ ]:
iso1_g6.var_names = iso1_g6.var["gene_symbol"].astype(str).values
iso1_g6.var_names_make_unique()

In [ ]:
sc.pl.violin(
    iso1_g6,
    keys=["Htr2a"],          # now it exists as a var_name
    groupby="group_6",
    stripplot=False,
    rotation=30,
)

In [ ]:
sc.pl.violin(
    iso1_g6,
    keys=["Htr2a"],          # now it exists as a var_name
    groupby="subclass",
    stripplot=False,
    rotation=30,
)

In [ ]:
sc.pl.dotplot(
    iso1_g6,
    var_names=["Htr2a"],
    groupby="group_6",
    gene_symbols="gene_symbol"
)

In [ ]:
iso1_g6

In [ ]:
frac = (
    iso1_g6.obs.groupby("group_6")["Htr2a_pos"]
    .mean()
    .sort_values(ascending=False)
)

frac

In [ ]:
ax = frac.plot(kind="bar")
ax.set_ylabel("Fraction Htr2a-positive")
ax.set_xlabel("")

In [ ]:
#summary[summary["n"] > 100].head(40)

In [ ]:
iso1_g6.obs[["class", "group_6"]].value_counts()

In [ ]:
iso1_g6.obs["Htr2a_pos"]

In [ ]:
iso1_g6_expr = iso1_g6[iso1_g6.obs["Htr2a_pos"]].copy()

In [ ]:
idxs = np.where(iso1_g6.var["gene_symbol"].values == "Htr2a")[0]
print("Htr2a matches:", len(idxs))

Xg = iso1_g6.X[:, idxs]

htr2a_sum = np.asarray(Xg.sum(axis=1)).ravel() if sparse.issparse(Xg) else Xg.sum(axis=1)

iso1_g6.obs["Htr2a_sum"] = htr2a_sum
iso1_g6.obs["Htr2a_pos"] = htr2a_sum > 0

In [ ]:
sc.pl.violin(
    iso1_g6,
    keys=["Htr2a_sum"],
    groupby="group_6",
    stripplot=False,
    rotation=30,
)

In [ ]:
iso1_g6_expr = iso1_g6[iso1_g6.obs["Htr2a_pos"]].copy()

sc.pl.violin(
    iso1_g6_expr,
    keys=["Htr2a_sum"],
    groupby="group_6",
    stripplot=False,
    rotation=30,
)

In [ ]:
iso1_g6.obs["Htr2a_sum_log1p"] = np.log1p(iso1_g6.obs["Htr2a_sum"].values)

sc.pl.violin(
    iso1_g6,
    keys=["Htr2a_sum_log1p"],
    groupby="group_6",
    stripplot=False,
    rotation=30,
)

In [ ]:
iso1_g6

In [ ]:
iso1_g6_expr